# Chapman-Shaoxing Preprocessing — Native

Extracts hand-crafted features from the **Chapman-Shaoxing 12-lead ECG database (45K patients)** to match the PTB-XL feature schema exactly.

## Sources

- **Zheng et al. (2020).** A 12-lead electrocardiogram database for arrhythmia research covering more than 10,000 patients. *Scientific Data* 7:48. DOI: 10.1038/s41597-020-0386-x
- **Zheng et al. (2022).** A large scale 12-lead electrocardiogram database for arrhythmia study (version 1.0.0). *PhysioNet*. DOI: 10.13026/wgex-er52
- **Goldberger et al. (2000).** *Circulation* 101(23):e215–e220 (PhysioResource attribution)

## Decisions and citations

| Decision | Choice | Justification |
|---|---|---|
| Sampling rate | Resample to 100 Hz from native 500 Hz | Match PTB-XL pipeline (Strodthoff et al. 2021) |
| AF labels | SNOMED 164889003 (AFIB) ∪ 164890007 (AFL), key presence | Mirrors PTB-XL AFIB ∪ AFLT logic |
| Feature library | NeuroKit2 (same as PTB-XL) | Cross-dataset feature comparability |
| Splits | New `strat_fold` assignment (1–10), patient-disjoint, AF-stratified | Patient-disjoint stratification per Wagner et al. 2020 protocol |
| Quality filter | None at extraction | Decision deferred to downstream |

## Output

`data/chapman_native_features.csv` — drop-in compatible with PTB-XL feature schema:
- `ecg_id`, `patient_id`, `strat_fold`, `validated_by_human` (always False — Chapman has no human re-validation flag)
- 4 demographics: `age`, `sex`, `height`, `weight`
- 5 cardiac intervals: `heart_rate`, `rr_interval_mean`, `rr_interval_std`, `pr_interval`, `qrs_duration`
- 5 HRV: `hrv_rmssd`, `hrv_sdnn`, `hrv_pnn50`, `hrv_mean_nn`, `hrv_sample_entropy`
- 48 lead stats: 12 leads × {mean, std, max, min}
- `af_detected`

**Total: 62 features + 4 IDs/metadata + 1 label = 67 columns** (matches PTB-XL schema exactly).

## 1. Imports and configuration

In [ ]:
import re
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk
from scipy.signal import resample_poly
from sklearn.model_selection import StratifiedGroupKFold
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Configuration
CHAPMAN_DIR = Path('data/chapman_shaoxing/')
WFDB_ROOT = CHAPMAN_DIR / 'WFDBRecords'
RECORDS_LIST = CHAPMAN_DIR / 'RECORDS'
CONDITION_NAMES = CHAPMAN_DIR / 'ConditionNames_SNOMED-CT.csv'
OUTPUT_PATH = Path('data/chapman_native_features.csv')
FAILURE_LOG_PATH = Path('data/chapman_extraction_failures.csv')

TARGET_FS = 100  # Match PTB-XL pipeline
NATIVE_FS = 500  # Chapman native sampling rate

# SNOMED CT codes for AF
# 164889003 = Atrial fibrillation
# 164890007 = Atrial flutter
AF_SNOMED_CODES = {'164889003', '164890007'}

# Splits
N_FOLDS = 10
FOLD_SEED = 2026

LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
         'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('chapman_prep')

## 2. Sanity check the dataset structure

Before parsing 45K records, verify the directory layout matches expectations.

In [ ]:
assert CHAPMAN_DIR.exists(), f'Chapman directory not found: {CHAPMAN_DIR.resolve()}'
assert WFDB_ROOT.exists(), f'WFDBRecords directory not found: {WFDB_ROOT.resolve()}'
assert RECORDS_LIST.exists(), f'RECORDS file not found: {RECORDS_LIST.resolve()}'

# RECORDS lists leaf subfolder paths like 'WFDBRecords/01/010/'
# We need to walk each folder and discover the individual .hea files
with open(RECORDS_LIST) as f:
    folder_paths = [line.strip().rstrip('/') for line in f if line.strip()]

log.info(f'RECORDS file lists {len(folder_paths)} leaf folders')
log.info(f'First 3: {folder_paths[:3]}')

# Strip the 'WFDBRecords/' prefix since CHAPMAN_DIR already contains it
def strip_wfdb_prefix(p):
    if p.startswith('WFDBRecords/'):
        return p[len('WFDBRecords/'):]
    if p.startswith('WFDBRecords\\'):
        return p[len('WFDBRecords\\'):]
    return p

folder_paths = [strip_wfdb_prefix(p) for p in folder_paths]
log.info(f'After stripping prefix: first 3 = {folder_paths[:3]}')

# Discover all .hea files within each leaf folder
record_paths = []
for folder_rel in folder_paths:
    folder_full = WFDB_ROOT / folder_rel
    if not folder_full.exists():
        log.warning(f'Folder listed in RECORDS not found: {folder_full}')
        continue
    for hea_file in sorted(folder_full.glob('*.hea')):
        # Build relative path: '01/010/JS00001' (no .hea extension, no WFDBRecords/ prefix)
        rel_stem = f'{folder_rel}/{hea_file.stem}'
        record_paths.append(rel_stem)

log.info(f'Discovered {len(record_paths)} total records across {len(folder_paths)} folders')
log.info(f'First 3 record paths: {record_paths[:3]}')

# Verify first record exists
first_path = WFDB_ROOT / record_paths[0]
hea_file = first_path.parent / f'{first_path.name}.hea'
assert hea_file.exists(), f'First record header not found: {hea_file}'
log.info(f'First record header verified: {hea_file}')

# Peek at the header
with open(hea_file) as f:
    print(f.read())

14:00:14 [INFO] RECORDS file lists 452 leaf folders
14:00:14 [INFO] First 3: ['WFDBRecords/01/010', 'WFDBRecords/01/011', 'WFDBRecords/01/012']
14:00:14 [INFO] After stripping prefix: first 3 = ['01/010', '01/011', '01/012']
14:00:14 [INFO] Discovered 45152 total records across 452 folders
14:00:14 [INFO] First 3 record paths: ['01/010/JS00001', '01/010/JS00002', '01/010/JS00004']
14:00:14 [INFO] First record header verified: data/chapman_shaoxing/WFDBRecords/01/010/JS00001.hea


JS00001 12 500 5000
JS00001.mat 16+24 1000/mV 16 0 -254 21756 0 I
JS00001.mat 16+24 1000/mV 16 0 264 -599 0 II
JS00001.mat 16+24 1000/mV 16 0 517 -22376 0 III
JS00001.mat 16+24 1000/mV 16 0 -5 28232 0 aVR
JS00001.mat 16+24 1000/mV 16 0 -386 16619 0 aVL
JS00001.mat 16+24 1000/mV 16 0 390 15121 0 aVF
JS00001.mat 16+24 1000/mV 16 0 -98 1568 0 V1
JS00001.mat 16+24 1000/mV 16 0 -312 -32761 0 V2
JS00001.mat 16+24 1000/mV 16 0 -98 32715 0 V3
JS00001.mat 16+24 1000/mV 16 0 810 15193 0 V4
JS00001.mat 16+24 1000/mV 16 0 810 14081 0 V5
JS00001.mat 16+24 1000/mV 16 0 527 32579 0 V6
#Age: 85
#Sex: Male
#Dx: 164889003,59118001,164934002
#Rx: Unknown
#Hx: Unknown
#Sx: Unknown



## 3. Parse a single header file

Chapman `.hea` files have:
- Standard WFDB header lines (record name, n_leads, sampling rate, n_samples, lead specs)
- Comment lines starting with `#` containing demographics and SNOMED codes:
  - `#Age: 85`
  - `#Sex: Male` / `#Sex: Female`
  - `#Dx: 164889003,427084000,...` (comma-separated SNOMED codes)
  - `#Rx: ...`, `#Hx: ...`, `#Sx: ...` (other clinical fields, not used)

**Note:** Chapman does not record height or weight. We set both to NaN to maintain the PTB-XL schema; downstream median-imputation will handle these consistently.

In [ ]:
def parse_chapman_header(hea_path):
    """Parse demographics and SNOMED codes from a Chapman .hea file.
    
    Returns dict with: age, sex (0=female, 1=male), dx_codes (set of strings).
    Returns None if the file can't be opened.
    """
    out = {'age': np.nan, 'sex': np.nan, 'dx_codes': set()}
    
    try:
        with open(hea_path) as f:
            lines = f.read().splitlines()
    except Exception:
        return None
    
    for line in lines:
        if not line.startswith('#'):
            continue
        if line.startswith('#Age:'):
            m = re.search(r'#Age:\s*(\d+)', line)
            if m:
                out['age'] = float(m.group(1))
        elif line.startswith('#Sex:'):
            sex_str = line.replace('#Sex:', '').strip().lower()
            if sex_str.startswith('m'):
                out['sex'] = 1  # Match PTB-XL convention: male=1
            elif sex_str.startswith('f'):
                out['sex'] = 0  # female=0
        elif line.startswith('#Dx:'):
            codes_str = line.replace('#Dx:', '').strip()
            codes = {c.strip() for c in codes_str.split(',') if c.strip()}
            out['dx_codes'] = codes
    
    return out


# Self-test on first record
_test_path = WFDB_ROOT / f'{record_paths[0]}.hea'
_parsed = parse_chapman_header(_test_path)
assert _parsed is not None, 'Header parse returned None'
log.info(f'First record parse: age={_parsed["age"]}, sex={_parsed["sex"]}, n_dx={len(_parsed["dx_codes"])}')
log.info(f'First record SNOMED codes: {_parsed["dx_codes"]}')

14:00:14 [INFO] First record parse: age=85.0, sex=1, n_dx=3
14:00:14 [INFO] First record SNOMED codes: {'164889003', '59118001', '164934002'}


## 4. Patient ID extraction

Chapman record IDs are `JS00001`, `JS00002`, etc. Per Zheng et al. 2020 *Scientific Data*, **each record corresponds to a unique patient** (one ECG per patient in the original Chapman 10K release). The PhysioNet 45K version aggregates multiple sources but maintains one-record-per-patient.

We use the record ID itself as the patient ID. This means in patient-disjoint splits, each fold contains different `JS*` IDs.

In [ ]:
def get_patient_id(record_path_str):
    """Extract patient ID from a record path like '01/010/JS00001'.
    
    Returns the JS-prefixed ID. Each Chapman record = one patient.
    """
    name = Path(record_path_str).name  # e.g. 'JS00001'
    return name


# Self-test
assert get_patient_id('01/010/JS00001') == 'JS00001'
log.info('Patient ID extraction self-test passed')

14:00:14 [INFO] Patient ID extraction self-test passed


## 5. Feature extraction (matching PTB-XL pipeline)

Same logic as PTB-XL preprocessor:
- Lead II → NeuroKit2 → heart rate, RR, PR, QRS, HRV bundle
- All 12 leads → mean/std/max/min

**Difference from PTB-XL:** Chapman is recorded at 500 Hz native; we resample to 100 Hz to match PTB-XL pipeline. Resampling is done before feature extraction so all downstream timing parameters (R-peak detection windows, etc.) are consistent.

In [ ]:
def extract_intervals_and_hrv(signal_lead_ii, fs):
    """Extract cardiac intervals + HRV features from Lead II.
    
    Identical implementation to the PTB-XL preprocessor for cross-dataset consistency.
    Returns dict with 10 features (5 intervals + 5 HRV).
    """
    nan_result = {
        'heart_rate': np.nan,
        'rr_interval_mean': np.nan,
        'rr_interval_std': np.nan,
        'pr_interval': np.nan,
        'qrs_duration': np.nan,
        'hrv_rmssd': np.nan,
        'hrv_sdnn': np.nan,
        'hrv_pnn50': np.nan,
        'hrv_mean_nn': np.nan,
        'hrv_sample_entropy': np.nan,
    }
    
    try:
        signals_df, info = nk.ecg_process(signal_lead_ii, sampling_rate=fs)
        rpeaks = info.get('ECG_R_Peaks', np.array([]))
    except Exception as e:
        return nan_result, f'ecg_process: {type(e).__name__}'
    
    out = dict(nan_result)
    
    # Heart rate from ECG_Rate column of signals_df (per PTB-XL preprocessor)
    if 'ECG_Rate' in signals_df.columns:
        rate_series = signals_df['ECG_Rate'].dropna()
        if len(rate_series) > 0:
            out['heart_rate'] = float(rate_series.mean())
    
    if len(rpeaks) >= 2:
        rr_ms = np.diff(rpeaks) / fs * 1000.0
        out['rr_interval_mean'] = float(rr_ms.mean())
        out['rr_interval_std'] = float(rr_ms.std())
        
        if len(rr_ms) >= 4:
            try:
                hrv_time = nk.hrv_time({'ECG_R_Peaks': rpeaks}, sampling_rate=fs)
                if 'HRV_RMSSD' in hrv_time.columns:
                    out['hrv_rmssd'] = float(hrv_time['HRV_RMSSD'].iloc[0])
                if 'HRV_SDNN' in hrv_time.columns:
                    out['hrv_sdnn'] = float(hrv_time['HRV_SDNN'].iloc[0])
                if 'HRV_pNN50' in hrv_time.columns:
                    out['hrv_pnn50'] = float(hrv_time['HRV_pNN50'].iloc[0])
                if 'HRV_MeanNN' in hrv_time.columns:
                    out['hrv_mean_nn'] = float(hrv_time['HRV_MeanNN'].iloc[0])
            except Exception:
                pass
            
            try:
                samp_en, _ = nk.entropy_sample(rr_ms, dimension=2, tolerance=0.2 * rr_ms.std())
                if not np.isnan(samp_en):
                    out['hrv_sample_entropy'] = float(samp_en)
            except Exception:
                pass
    
    if len(rpeaks) >= 2:
        try:
            _, waves = nk.ecg_delineate(
                signal_lead_ii, rpeaks, sampling_rate=fs, method='dwt'
            )
            
            p_on = np.array(waves.get('ECG_P_Onsets', []), dtype=float)
            r_on = np.array(waves.get('ECG_R_Onsets', []), dtype=float)
            r_off = np.array(waves.get('ECG_R_Offsets', []), dtype=float)
            
            p_on = p_on[~np.isnan(p_on)]
            r_on = r_on[~np.isnan(r_on)]
            r_off = r_off[~np.isnan(r_off)]
            
            if len(p_on) > 0 and len(r_on) > 0:
                n = min(len(p_on), len(r_on))
                pr_ms = (r_on[:n] - p_on[:n]) / fs * 1000.0
                pr_ms = pr_ms[(pr_ms > 80) & (pr_ms < 320)]
                if len(pr_ms) > 0:
                    out['pr_interval'] = float(pr_ms.mean())
            
            if len(r_on) > 0 and len(r_off) > 0:
                n = min(len(r_on), len(r_off))
                qrs_ms = (r_off[:n] - r_on[:n]) / fs * 1000.0
                qrs_ms = qrs_ms[(qrs_ms > 40) & (qrs_ms < 200)]
                if len(qrs_ms) > 0:
                    out['qrs_duration'] = float(qrs_ms.mean())
        except Exception:
            pass
    
    return out, None


def extract_lead_statistics(signals):
    """Extract per-lead amplitude statistics. signals: (n_samples, 12)."""
    out = {}
    for i, lead in enumerate(LEADS):
        sig = signals[:, i]
        out[f'{lead}_mean'] = float(sig.mean())
        out[f'{lead}_std'] = float(sig.std())
        out[f'{lead}_max'] = float(sig.max())
        out[f'{lead}_min'] = float(sig.min())
    return out


def resample_to_target(signals, native_fs, target_fs):
    """Resample (n_samples, n_leads) signal from native_fs to target_fs using polyphase filter."""
    if native_fs == target_fs:
        return signals
    # 500 → 100 means up=1, down=5
    from math import gcd
    g = gcd(int(native_fs), int(target_fs))
    up = int(target_fs // g)
    down = int(native_fs // g)
    return resample_poly(signals, up, down, axis=0)

## 6. Lead-order verification

Chapman uses standard 12-lead ordering matching PTB-XL: I, II, III, aVR, aVL, aVF, V1–V6. Verify on first record.

In [ ]:
_first_rec = wfdb.rdrecord(str(WFDB_ROOT / record_paths[0]))
log.info(f'First record signal shape: {_first_rec.p_signal.shape}')
log.info(f'First record sig_name: {_first_rec.sig_name}')
log.info(f'First record fs: {_first_rec.fs}')

# Verify lead order matches our LEADS constant (case-insensitive match)
actual_leads = [s.upper() for s in _first_rec.sig_name]
expected_leads_upper = [l.upper() for l in LEADS]

if actual_leads == expected_leads_upper:
    log.info('Lead order matches PTB-XL convention exactly')
else:
    log.warning(f'Lead order mismatch! Actual: {actual_leads}')
    log.warning(f'                    Expected: {expected_leads_upper}')
    log.warning('Will use index-based reordering during extraction')

# Build lead reordering map
lead_index_map = {}
for target_idx, target_lead in enumerate(expected_leads_upper):
    if target_lead in actual_leads:
        lead_index_map[target_idx] = actual_leads.index(target_lead)
    else:
        log.error(f'Lead {target_lead} not found in record!')
        raise ValueError(f'Missing lead: {target_lead}')
log.info(f'Lead index map: {lead_index_map}')

14:00:14 [INFO] First record signal shape: (5000, 12)
14:00:14 [INFO] First record sig_name: ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
14:00:14 [INFO] First record fs: 500
14:00:14 [INFO] Lead order matches PTB-XL convention exactly
14:00:14 [INFO] Lead index map: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11}


## 7. Main extraction loop

In [ ]:
def extract_one_record(record_path_relative, lead_index_map):
    """Extract features from a single Chapman record.
    
    Args:
        record_path_relative: e.g. '01/010/JS00001'
        lead_index_map: dict mapping target lead index -> actual sig_name index
    
    Returns:
        (feature_dict, failure_record_or_None)
    """
    full_path = WFDB_ROOT / record_path_relative
    hea_path = full_path.parent / f'{full_path.name}.hea'
    
    # Initialize feature dict
    feature_dict = {
        'ecg_id': full_path.name,
        'patient_id': get_patient_id(record_path_relative),
        'strat_fold': -1,  # filled in later
        'validated_by_human': False,  # Chapman does not provide this flag
        'age': np.nan,
        'sex': np.nan,
        'height': np.nan,  # Chapman does not record height
        'weight': np.nan,  # Chapman does not record weight
        'af_detected': 0,
    }
    # Initialize all waveform features as NaN
    for f in ['heart_rate', 'rr_interval_mean', 'rr_interval_std',
              'pr_interval', 'qrs_duration',
              'hrv_rmssd', 'hrv_sdnn', 'hrv_pnn50',
              'hrv_mean_nn', 'hrv_sample_entropy']:
        feature_dict[f] = np.nan
    for lead in LEADS:
        for stat in ['mean', 'std', 'max', 'min']:
            feature_dict[f'{lead}_{stat}'] = np.nan
    
    # Parse header
    parsed = parse_chapman_header(hea_path)
    if parsed is None:
        return feature_dict, {
            'ecg_id': full_path.name,
            'af_detected': 0,
            'stage': 'header_parse',
            'reason': f'cannot open {hea_path}',
        }
    
    feature_dict['age'] = parsed['age']
    feature_dict['sex'] = parsed['sex']
    
    # AF label
    af_label = int(bool(AF_SNOMED_CODES & parsed['dx_codes']))
    feature_dict['af_detected'] = af_label
    
    # Load and process signal
    try:
        record = wfdb.rdrecord(str(full_path))
        signals_native = record.p_signal  # (n_samples, 12)
        native_fs = record.fs
        
        # Reorder leads to PTB-XL convention if needed
        if lead_index_map:
            n_leads = len(LEADS)
            reorder_idx = [lead_index_map[i] for i in range(n_leads)]
            signals_native = signals_native[:, reorder_idx]
        
        # Resample to 100 Hz
        signals = resample_to_target(signals_native, native_fs, TARGET_FS)
        
        # Tier 2 + 2b: Lead II intervals + HRV at TARGET_FS
        ii_features, ii_err = extract_intervals_and_hrv(signals[:, 1], TARGET_FS)
        feature_dict.update(ii_features)
        
        # Tier 3: lead amplitude stats
        feature_dict.update(extract_lead_statistics(signals))
        
        if ii_err:
            return feature_dict, {
                'ecg_id': full_path.name,
                'af_detected': af_label,
                'stage': 'intervals_hrv',
                'reason': ii_err,
            }
        
        return feature_dict, None
    
    except FileNotFoundError:
        return feature_dict, {
            'ecg_id': full_path.name,
            'af_detected': af_label,
            'stage': 'file_load',
            'reason': f'FileNotFound: {full_path}',
        }
    except Exception as e:
        return feature_dict, {
            'ecg_id': full_path.name,
            'af_detected': af_label,
            'stage': 'waveform_processing',
            'reason': f'{type(e).__name__}: {str(e)[:100]}',
        }


def extract_all(record_paths, lead_index_map, sample_limit=None):
    """Extract features from all Chapman records."""
    if sample_limit:
        record_paths = record_paths[:sample_limit]
        log.info(f'Sample limit: processing first {sample_limit} records')
    
    features_list = []
    failures = []
    
    for rec_path in tqdm(record_paths, desc='Extracting'):
        feat, fail = extract_one_record(rec_path, lead_index_map)
        features_list.append(feat)
        if fail is not None:
            failures.append(fail)
    
    return pd.DataFrame(features_list), pd.DataFrame(failures)

## 8. Smoke test on 200 records

In [ ]:
log.info('=== SMOKE TEST: 200 records ===')
df_test, failures_test = extract_all(record_paths, lead_index_map, sample_limit=200)

log.info(f'Test DataFrame shape: {df_test.shape}')
log.info(f'Test failures: {len(failures_test)} ({100*len(failures_test)/len(df_test):.1f}%)')

# Schema check
expected_columns = (
    ['ecg_id', 'patient_id', 'strat_fold', 'validated_by_human']
    + ['age', 'sex', 'height', 'weight']
    + ['heart_rate', 'rr_interval_mean', 'rr_interval_std', 'pr_interval', 'qrs_duration']
    + ['hrv_rmssd', 'hrv_sdnn', 'hrv_pnn50', 'hrv_mean_nn', 'hrv_sample_entropy']
    + [f'{l}_{s}' for l in LEADS for s in ['mean', 'std', 'max', 'min']]
    + ['af_detected']
)
missing = set(expected_columns) - set(df_test.columns)
extra = set(df_test.columns) - set(expected_columns)
assert not missing, f'Missing columns: {missing}'
assert not extra, f'Unexpected extra columns: {extra}'
log.info(f'Schema OK: {len(expected_columns)} columns')

# Spot-check populated features
log.info(f'AF prevalence in test sample: {df_test["af_detected"].mean():.4f}')
log.info(f'Records with valid heart_rate: {df_test["heart_rate"].notna().sum()}/{len(df_test)}')
log.info(f'Records with valid hrv_rmssd: {df_test["hrv_rmssd"].notna().sum()}/{len(df_test)}')
if df_test['hrv_rmssd'].notna().sum() > 0:
    log.info(f'RMSSD range: {df_test["hrv_rmssd"].min():.1f} – {df_test["hrv_rmssd"].max():.1f} ms')

if len(failures_test) > 0:
    log.info(f'Failure stages:')
    print(failures_test['stage'].value_counts().to_string())

14:00:14 [INFO] === SMOKE TEST: 200 records ===
14:00:14 [INFO] Sample limit: processing first 200 records
Extracting: 100%|████████████████████████████████████████████████████████████| 200/200 [00:10<00:00, 19.18it/s]
14:00:24 [INFO] Test DataFrame shape: (200, 67)
14:00:24 [INFO] Test failures: 0 (0.0%)
14:00:24 [INFO] Schema OK: 67 columns
14:00:24 [INFO] AF prevalence in test sample: 0.2450
14:00:24 [INFO] Records with valid heart_rate: 200/200
14:00:24 [INFO] Records with valid hrv_rmssd: 200/200
14:00:24 [INFO] RMSSD range: 5.8 – 719.3 ms


## 9. Full extraction

If smoke test passes, extract all 45,152 records. Expect ~45–90 minutes depending on hardware.

In [ ]:
smoke_ok = (
    len(df_test) >= 180
    and len(failures_test) / len(df_test) < 0.10
    and df_test['heart_rate'].notna().sum() > 100
)
assert smoke_ok, (
    'Smoke test failed: high failure rate or too few HR extractions. '
    'Inspect failures_test before proceeding.'
)
log.info('Smoke test passed. Running full extraction...')

df_full, failures_full = extract_all(record_paths, lead_index_map)

log.info(f'Full extraction complete')
log.info(f'Records processed: {len(df_full)}')
log.info(f'Total failures: {len(failures_full)} ({100*len(failures_full)/len(df_full):.2f}%)')

14:00:24 [INFO] Smoke test passed. Running full extraction...
Extracting: 100%|████████████████████████████████████████████████████████| 45152/45152 [42:37<00:00, 17.65it/s]
14:43:02 [INFO] Full extraction complete
14:43:02 [INFO] Records processed: 45152
14:43:02 [INFO] Total failures: 396 (0.88%)


## 10. Assign patient-disjoint, AF-stratified folds (1–10)

Same protocol as PTB-XL: 10 folds, stratified by AF label, patient-disjoint. Since each Chapman record = one patient, this is equivalent to a stratified k-fold split.

Convention: folds 1–8 train, 9 validation, 10 test (Wagner et al. 2020).

In [ ]:
skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED)

y = df_full['af_detected'].values
groups = df_full['patient_id'].values
X_dummy = np.zeros((len(df_full), 1))

fold_assignments = np.zeros(len(df_full), dtype=int)
for fold_idx, (_, test_idx) in enumerate(skf.split(X_dummy, y, groups)):
    fold_assignments[test_idx] = fold_idx + 1  # 1-indexed: folds 1–10

df_full['strat_fold'] = fold_assignments

fold_counts = df_full['strat_fold'].value_counts().sort_index()
log.info(f'Fold distribution:\n{fold_counts.to_string()}')

# Verify stratification
fold_prev = df_full.groupby('strat_fold')['af_detected'].mean()
log.info(f'AF prevalence by fold:\n{fold_prev.round(4).to_string()}')

# Verify patient-disjoint (since each record is a unique patient, this is trivially true)
assert df_full['patient_id'].nunique() == len(df_full), \
    'Found duplicate patient IDs — patient-disjoint property violated'
log.info('Patient-disjoint verified (one record per patient)')

14:43:09 [INFO] Fold distribution:
strat_fold
1     4516
2     4515
3     4515
4     4515
5     4516
6     4515
7     4515
8     4515
9     4515
10    4515
14:43:09 [INFO] AF prevalence by fold:
strat_fold
1     0.2179
2     0.2179
3     0.2179
4     0.2179
5     0.2179
6     0.2179
7     0.2179
8     0.2179
9     0.2179
10    0.2179
14:43:09 [INFO] Patient-disjoint verified (one record per patient)


## 11. Sanity checks

In [ ]:
# Record count
n = len(df_full)
log.info(f'Record count: {n} (expected ~45,152)')
assert 40000 < n < 46000, f'Record count {n} far from expected 45,152'

# AF prevalence
prevalence = df_full['af_detected'].mean()
log.info(f'AF prevalence: {prevalence:.4f}')
# Chapman literature reports AF prevalence in the 18–25% range across various subsets
assert 0.10 < prevalence < 0.30, f'AF prevalence {prevalence:.4f} outside expected Chapman range [0.10, 0.30]'

# Fold balance
max_fold_dev = (fold_prev - prevalence).abs().max()
log.info(f'Max fold prevalence deviation: {max_fold_dev:.4f}')
assert max_fold_dev < 0.02, f'Folds not stratified properly: max dev {max_fold_dev:.4f}'

# Class-conditional failure rate
if len(failures_full) > 0:
    fail_by_class = failures_full.groupby('af_detected').size()
    af_records = (df_full['af_detected'] == 1).sum()
    nonaf_records = (df_full['af_detected'] == 0).sum()
    af_fail_rate = fail_by_class.get(1, 0) / max(af_records, 1)
    nonaf_fail_rate = fail_by_class.get(0, 0) / max(nonaf_records, 1)
    log.info(f'Failure rate by class: AF={af_fail_rate:.4f}, non-AF={nonaf_fail_rate:.4f}')
    if af_fail_rate > 2 * nonaf_fail_rate:
        log.warning(f'AF failure rate elevated — check for class-conditional bias')

log.info('All sanity checks passed.')

14:43:09 [INFO] Record count: 45152 (expected ~45,152)
14:43:09 [INFO] AF prevalence: 0.2179
14:43:09 [INFO] Max fold prevalence deviation: 0.0000
14:43:09 [INFO] Failure rate by class: AF=0.0160, non-AF=0.0068
14:43:09 [WARNING] AF failure rate elevated — check for class-conditional bias
14:43:09 [INFO] All sanity checks passed.


## 12. Feature summary

In [ ]:
feature_cols = [c for c in df_full.columns
                if c not in ('ecg_id', 'patient_id', 'strat_fold', 'validated_by_human', 'af_detected')]
log.info(f'Total features: {len(feature_cols)}')

missing_pct = (df_full[feature_cols].isna().sum() / len(df_full) * 100).sort_values(ascending=False)
log.info('Top 10 features by missing rate:')
print(missing_pct.head(10).round(2).to_string())

# Note: height/weight will be 100% missing — Chapman does not record these
for col in ('height', 'weight'):
    if missing_pct.get(col, 0) > 99:
        log.info(f'Note: {col} is 100% missing (Chapman does not record this)')

14:43:09 [INFO] Total features: 62
14:43:09 [INFO] Top 10 features by missing rate:
14:43:09 [INFO] Note: height is 100% missing (Chapman does not record this)
14:43:09 [INFO] Note: weight is 100% missing (Chapman does not record this)


height                100.00
weight                100.00
qrs_duration           13.16
pr_interval            12.42
hrv_rmssd               1.01
hrv_sdnn                1.01
hrv_pnn50               1.01
hrv_mean_nn             1.01
hrv_sample_entropy      1.01
heart_rate              0.88


## 13. Save outputs

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_full.to_csv(OUTPUT_PATH, index=False)
log.info(f'Saved features to {OUTPUT_PATH} ({df_full.shape[0]} rows × {df_full.shape[1]} cols)')

if len(failures_full) > 0:
    failures_full.to_csv(FAILURE_LOG_PATH, index=False)
    log.info(f'Saved failure log to {FAILURE_LOG_PATH} ({len(failures_full)} failures)')

log.info('Done.')

14:43:10 [INFO] Saved features to data/chapman_native_features.csv (45152 rows × 67 cols)
14:43:10 [INFO] Saved failure log to data/chapman_extraction_failures.csv (396 failures)
14:43:10 [INFO] Done.
